In [1]:
# 05 - Stratified K-Fold Cross Validation
# Trains the CNN k times on different folds of the REAL (non-augmented) images
# to get a more reliable accuracy estimate than one single train/val split.
# Test set is never touched here - k-fold only uses the train+val pool.

import sys
sys.path.insert(0, '..')

import os
import yaml
import statistics
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import DataLoader

from src.models.cnn_model import SimpleCNN
from src.data.cross_validation import ListDataset, get_real_images, make_folds

In [2]:
# check device - use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
# reuse the winning learning rate and batch size from the main CNN config
with open('../configs/cnn_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

image_size = config['image_size']
batch_size = config['batch_size']
learning_rate = config['learning_rate']

# fewer max epochs per fold - training happens 5 separate times here,
# early stopping cuts each one off once it stops improving anyway
max_epochs = 20

transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor()
])

print("Using:", config, "max_epochs per fold:", max_epochs)

Using: {'batch_size': 16, 'epochs': 40, 'image_size': 128, 'learning_rate': 0.001} max_epochs per fold: 20


In [4]:
# gather all real images (train + val combined, augmented files skipped)
image_paths, labels, class_names = get_real_images('../data/processed')

print("Total real images for cross validation:", len(image_paths))
print("Classes:", class_names)

Total real images for cross validation: 347
Classes: ['healthy', 'low_tread', 'sidewall_damaged', 'uneven_wear', 'zero_tread']


In [5]:
# build 5 stratified folds - same class ratio preserved in every fold
k = 5
folds = make_folds(image_paths, labels, k=k, seed=42)

print("Made", len(folds), "folds")

Made 5 folds


In [6]:
# class weights for this fold's training data
# used instead of re-augmenting each fold - gives rare classes more weight
# in the loss so the model doesn't just learn to guess the majority class

def get_class_weights(labels_subset, class_names):
    counts = []
    for class_name in class_names:
        count = labels_subset.count(class_name)
        counts.append(count)

    counts = torch.tensor(counts, dtype=torch.float)
    weights = 1.0 / counts
    weights = weights / weights.sum()
    return weights

In [7]:
fold_accuracies = []

for fold_number in range(k):
    train_index, val_index = folds[fold_number]

    print()
    print("=== Fold", fold_number + 1, "of", k, "===")

    fold_train_paths = [image_paths[i] for i in train_index]
    fold_train_labels = [labels[i] for i in train_index]
    fold_val_paths = [image_paths[i] for i in val_index]
    fold_val_labels = [labels[i] for i in val_index]

    train_dataset = ListDataset(fold_train_paths, fold_train_labels, class_names, transform)
    val_dataset = ListDataset(fold_val_paths, fold_val_labels, class_names, transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # fresh model every fold - no weights carried over between folds
    model = SimpleCNN(len(class_names)).to(device)

    class_weights = get_class_weights(fold_train_labels, class_names).to(device)
    loss_function = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    patience = 5
    epochs_without_improvement = 0
    best_val_acc = 0

    for epoch in range(max_epochs):
        model.train()
        for images, image_labels in train_loader:
            images = images.to(device)
            image_labels = image_labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = loss_function(outputs, image_labels)
            loss.backward()
            optimizer.step()

        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, image_labels in val_loader:
                images = images.to(device)
                image_labels = image_labels.to(device)
                outputs = model(images)
                predicted = outputs.argmax(dim=1)
                correct += (predicted == image_labels).sum().item()
                total += image_labels.size(0)

        val_acc = correct / total

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    print("Fold", fold_number + 1, "best val accuracy:", round(best_val_acc, 3))
    fold_accuracies.append(best_val_acc)


=== Fold 1 of 5 ===


KeyboardInterrupt: 

In [ ]:
# summary across all 5 folds
average_acc = statistics.mean(fold_accuracies)
std_acc = statistics.stdev(fold_accuracies)

print("Fold accuracies:", [round(a, 3) for a in fold_accuracies])
print("Average accuracy:", round(average_acc, 3))
print("Standard deviation:", round(std_acc, 3))

In [ ]:
# save the summary - small text file, goes in git as evidence
os.makedirs("../results/cnn", exist_ok=True)

with open("../results/cnn/kfold_results.txt", "w") as f:
    f.write("Stratified " + str(k) + "-fold cross validation results\n\n")
    f.write("Settings used: " + str(config) + "\n\n")
    for i, acc in enumerate(fold_accuracies):
        f.write("Fold " + str(i + 1) + ": " + str(round(acc, 3)) + "\n")
    f.write("\nAverage accuracy: " + str(round(average_acc, 3)) + "\n")
    f.write("Standard deviation: " + str(round(std_acc, 3)) + "\n")

print("Saved to results/cnn/kfold_results.txt")